In [61]:
# %pip install nltk #, pandas, numpy, matplotlib, scipy, scikit-learn

In [62]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from nltk.stem.snowball import SnowballStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.stem import PorterStemmer
from nltk.tokenize import sent_tokenize, word_tokenize
import re
from tqdm import tqdm
import spacy

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [63]:
# Define paths

movie_df_path = "../data/tmdb_movie_data_parsed.csv"

In [64]:
# Movie Content Data
movie_df = pd.read_csv(movie_df_path)

# Parse JSON on genres, keywords, crew, cast, production
movie_df['genres'] = movie_df['genres'].apply(literal_eval)
movie_df['keywords'] = movie_df['keywords'].apply(literal_eval)
movie_df['crew'] = movie_df['crew'].apply(literal_eval)
movie_df['cast'] = movie_df['cast'].apply(literal_eval)
movie_df['productions'] = movie_df['productions'].apply(literal_eval)

movie_df.head()

,budget,genres,id,revenue,runtime,title,year,weighted_rating,description,keywords,crew,cast,productions
0,30000000,"[Family, Comedy, Animation, Adventure]",862,401157969,81,Toy Story,1995,7.974369,"Led by Woody, Andy's toys live happily in his ...","[rescue, friendship, mission, jealousy, villai...","[writer_alec_sokolow, producer_edwin_catmull, ...","[actor_tom_hanks, actor_tim_allen, actor_don_r...",[production_pixar]
1,65000000,"[Adventure, Fantasy, Family]",8844,262821940,104,Jumanji,1995,7.243814,When siblings Judy and Peter discover an encha...,"[based on novel or book, giant insect, board g...","[writer_jim_strain, director_joe_johnston, pro...","[actor_robin_williams, actor_kirsten_dunst, ac...","[production_tristar_pictures, production_inter..."
2,25000000,"[Romance, Comedy]",15602,71518503,101,Grumpier Old Men,1995,6.482411,A family wedding reignites the ancient feud be...,"[fishing, sequel, old man, best friend, weddin...","[director_howard_deutch, writer_mark_steven_jo...","[actor_walter_matthau, actor_jack_lemmon, acto...","[production_lancaster_gate, production_warner_..."
3,16000000,"[Comedy, Drama, Romance]",31357,81452156,127,Waiting to Exhale,1995,6.297186,"Cheated on, mistreated and stepped on, the wom...","[based on novel or book, single mother, divorc...","[director_forest_whitaker, writer_terry_mcmill...","[actor_whitney_houston, actor_angela_bassett, ...",[production_20th_century_fox]
4,0,"[Comedy, Family]",11862,76594107,106,Father of the Bride Part II,1995,6.287352,Just when George Banks has recovered from his ...,"[daughter, baby, parent child relationship, mi...","[producer_nancy_meyers, writer_nancy_meyers, d...","[actor_steve_martin, actor_diane_keaton, actor...","[production_touchstone_pictures, production_sa..."


In [65]:
movie_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15711 entries, 0 to 15710
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   budget           15711 non-null  int64  
 1   genres           15711 non-null  object 
 2   id               15711 non-null  int64  
 3   revenue          15711 non-null  int64  
 4   runtime          15711 non-null  int64  
 5   title            15711 non-null  str    
 6   year             15711 non-null  int64  
 7   weighted_rating  15711 non-null  float64
 8   description      15711 non-null  str    
 9   keywords         15711 non-null  object 
 10  crew             15711 non-null  object 
 11  cast             15711 non-null  object 
 12  productions      15711 non-null  object 
dtypes: float64(1), int64(5), object(5), str(2)
memory usage: 1.6+ MB


In [66]:
from sklearn.preprocessing import MultiLabelBinarizer

genres_mlb = MultiLabelBinarizer()
genres_multihot = genres_mlb.fit_transform(movie_df["genres"])

# # save the multihot encoded genres to a new DataFrame
genres_df = pd.DataFrame(genres_multihot, columns=genres_mlb.classes_)
genres_df.head()
# concat_df = pd.concat([df, genres_df], axis=1)
# preprocessed_columns.update(["genres"])

# genres_multihot = movie_df['genres']

,Action,Adventure,Animation,Comedy,Crime,Documentary,Drama,Family,Fantasy,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
0,0,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
1,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0
4,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0


In [67]:
from sklearn.preprocessing import MinMaxScaler

nb_scaler = MinMaxScaler()

number_fields = ["budget", "revenue", "runtime", "year", "weighted_rating"]

nb_fields_scaled = nb_scaler.fit_transform(movie_df[number_fields])

nb_fields_scaled_df = pd.DataFrame(nb_fields_scaled, columns=number_fields)

nb_fields_scaled_df.head()

,budget,revenue,runtime,year,weighted_rating
0,0.042857,0.137209,0.138462,0.769231,0.876051
1,0.092857,0.089893,0.177778,0.769231,0.754513
2,0.035714,0.024462,0.172650,0.769231,0.627842
3,0.022857,0.027859,0.217094,0.769231,0.597027
4,0.000000,0.026198,0.181197,0.769231,0.595391


In [68]:
movie_df["keywords"].head()

0    [rescue, friendship, mission, jealousy, villai...
1    [based on novel or book, giant insect, board g...
2    [fishing, sequel, old man, best friend, weddin...
3    [based on novel or book, single mother, divorc...
4    [daughter, baby, parent child relationship, mi...
Name: keywords, dtype: object

In [69]:
ps = PorterStemmer()

In [70]:
lm = WordNetLemmatizer()
sb = SnowballStemmer('english')

In [71]:
sp = spacy.load('en_core_web_sm')

In [72]:
from collections import Counter
def tf(overview):
    vector_length=0
    overview_words=re.sub("[^\w'-]"," ", str(overview).lower()).split()
    stemmed_words=list()
    for word in overview_words:
        stemmed_words.append(word)
    overview_words=Counter(stemmed_words) 
    
    words_dicts=dict()
    for word,count in overview_words.items():
        vector_length+=((1+np.log10(count))**(2))
    vector_length=vector_length**(0.5)
    for word,count in overview_words.items():
        words_dicts.update({word:((1+np.log10(count))/vector_length)})
    return words_dicts

In [73]:
#pass a list of documents
def idf(idf_data): 
    idf_dict=dict()
    for docs in idf_data:
        doc_words=re.sub("[^\w'-]"," ", str(docs).lower()).split()
        stemmed_words=list()
        for word in doc_words:
            stemmed_words.append((word))
        
        doc_words=list(set(stemmed_words))
        for word in doc_words:
            if (word.lower()) not in idf_dict.keys():
                idf_dict.setdefault((word.lower()), 1)
            else:
                idf_dict[(word.lower())]+=1
    for key,value in idf_dict.items():
        idf_dict[key]=np.log10(len(idf_data)/value)
    return idf_dict

In [ ]:
keywords_column = list()
for movie_keywords in tqdm(movie_df["keywords"]):
    keywords = ''
    for d in movie_keywords:
        doc = sp(d)
        lemmatized = ' '.join([token.lemma_ for token in doc])
        keywords = keywords + lemmatized + ' '
    keywords_column.append(keywords.strip())

display(keywords_column)


  0%|          | 0/15711 [00:00<?, ?it/s]

100%|██████████| 15711/15711 [12:22<00:00, 21.16it/s]


['rescue friendship mission jealousy villain bully elementary school rivalry anthropomorphism friend computer animation buddy walkie talkie toy car boy next door new toy neighborhood toy come to life resourcefulness toy 3d animation joyous amuse comfort pixar',
 'base on novel or book giant insect board game disappearance jungle recluse stampede base on young adult novel excite joyful vibrant',
 'fishing sequel old man good friend wedding italian restaurant old friend duringcreditsstinger prank',
 'base on novel or book single mother divorce anxious friendship between woman cautionary adore celebratory comfort african american romance african american friendship',
 'daughter baby parent child relationship midlife crisis pregnancy confidence age sequel remake los angeles , california pregnant woman contraception gynecologist pregnant',
 'robbery chase obsession detective heist remake thief honor murder betrayal gang los angeles , california cat and mouse bank robbery criminal mastermind

In [75]:
keywords_df = pd.DataFrame(keywords_column, columns=['keywords'])

keywords_df
# TODO: Count Vectorizer

,keywords
0,rescue friendship mission jealousy villain bul...
1,base on novel or book giant insect board game ...
2,fishing sequel old man good friend wedding ita...
3,base on novel or book single mother divorce an...
4,daughter baby parent child relationship midlif...
...,...
15706,friendship tattoo rock star groupie past prom ...
15707,army escape bravery loyalty british empire isl...
15708,ransom kidnap hostage psychopath telephone maniac
15709,daughter upper class father murder musical fam...


In [76]:
# TODO: Lemma Description
description_columns = []
print("starting")

for description in tqdm(movie_df["description"]):
    doc = sp(description)
    # 1. Remove 's at the end of words
    # doc = re.sub(r"\b's\b", "", doc)

    # # 2. Remove double quotes surrounding words
    # doc = re.sub(r'"([^"\n]+)"', r"\1", doc)
    lemmatized = ' '.join([token.lemma_ for token in doc])
    description_columns.append(lemmatized.strip())

display(description_columns)

starting


100%|██████████| 15711/15711 [05:09<00:00, 50.80it/s]


["lead by Woody , Andy 's toy live happily in his room until Andy 's birthday bring Buzz Lightyear onto the scene . Afraid of lose his place in Andy 's heart , Woody plot against Buzz . but when circumstance separate Buzz and Woody from their owner , the duo eventually learn to put aside their difference . the adventure take off when toy come to life !",
 "when sibling Judy and Peter discover an enchanted board game that open the door to a magical world , they unwittingly invite Alan -- an adult who be be trap inside the game for 26 year -- into their living room . Alan 's only hope for freedom be to finish the game , which prove risky as all three find themselves run from giant rhinoceros , evil monkey and other terrifying creature . it be a jungle in here .",
 'a family wedding reignite the ancient feud between next - door neighbor and fishing buddy John and Max . meanwhile , a sultry italian divorcée open a restaurant at the local bait shop , alarm the local who worry she will scare

In [77]:
description_df = pd.DataFrame(description_columns, columns=['description'])

In [78]:
movie_df["keywords"] = keywords_df["keywords"]
movie_df["description"] = description_df["description"]

movie_df.to_csv("../data/tmdb_movie_data_parsed_progressed_lemmaed.csv", index=False)

movie_df.head()

,budget,genres,id,revenue,runtime,title,year,weighted_rating,description,keywords,crew,cast,productions
0,30000000,"[Family, Comedy, Animation, Adventure]",862,401157969,81,Toy Story,1995,7.974369,"lead by Woody , Andy 's toy live happily in hi...",rescue friendship mission jealousy villain bul...,"[writer_alec_sokolow, producer_edwin_catmull, ...","[actor_tom_hanks, actor_tim_allen, actor_don_r...",[production_pixar]
1,65000000,"[Adventure, Fantasy, Family]",8844,262821940,104,Jumanji,1995,7.243814,when sibling Judy and Peter discover an enchan...,base on novel or book giant insect board game ...,"[writer_jim_strain, director_joe_johnston, pro...","[actor_robin_williams, actor_kirsten_dunst, ac...","[production_tristar_pictures, production_inter..."
2,25000000,"[Romance, Comedy]",15602,71518503,101,Grumpier Old Men,1995,6.482411,a family wedding reignite the ancient feud bet...,fishing sequel old man good friend wedding ita...,"[director_howard_deutch, writer_mark_steven_jo...","[actor_walter_matthau, actor_jack_lemmon, acto...","[production_lancaster_gate, production_warner_..."
3,16000000,"[Comedy, Drama, Romance]",31357,81452156,127,Waiting to Exhale,1995,6.297186,"cheat on , mistreat and step on , the woman be...",base on novel or book single mother divorce an...,"[director_forest_whitaker, writer_terry_mcmill...","[actor_whitney_houston, actor_angela_bassett, ...",[production_20th_century_fox]
4,0,"[Comedy, Family]",11862,76594107,106,Father of the Bride Part II,1995,6.287352,just when George Banks have recover from his d...,daughter baby parent child relationship midlif...,"[producer_nancy_meyers, writer_nancy_meyers, d...","[actor_steve_martin, actor_diane_keaton, actor...","[production_touchstone_pictures, production_sa..."


In [79]:
description_df

,description
0,"lead by Woody , Andy 's toy live happily in hi..."
1,when sibling Judy and Peter discover an enchan...
2,a family wedding reignite the ancient feud bet...
3,"cheat on , mistreat and step on , the woman be..."
4,just when George Banks have recover from his d...
...,...
15706,"in the late ' 60 , the self - proclaim belle o..."
15707,a young british officer resign his post when h...
15708,when their daughter be abduct by experienced k...
15709,eight woman gather to celebrate Christmas in a...


In [80]:
tf = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    # min_df=0.003, 
    max_features=1500,
    stop_words='english'
    )

tfidf_matrix = tf.fit_transform(description_df["description"])
tfidf_matrix.shape


(15711, 1500)

In [81]:
list(i for i in tf.get_feature_names_out())

['000',
 '10',
 '11',
 '12',
 '12 year',
 '15',
 '16',
 '17',
 '1960',
 '1970',
 '1980',
 '19th',
 '19th century',
 '20',
 '20 year',
 '30',
 '40',
 '50',
 'abandon',
 'abduct',
 'ability',
 'able',
 'aboard',
 'abuse',
 'academy',
 'accept',
 'accident',
 'accidentally',
 'accompany',
 'account',
 'accuse',
 'achieve',
 'act',
 'action',
 'activity',
 'actor',
 'actress',
 'actually',
 'addict',
 'adopt',
 'adult',
 'adventure',
 'affair',
 'africa',
 'african',
 'age',
 'aged',
 'agent',
 'ago',
 'agree',
 'ahead',
 'aid',
 'air',
 'alex',
 'alice',
 'alien',
 'alive',
 'allow',
 'ally',
 'alter',
 'amazing',
 'ambitious',
 'america',
 'american',
 'americans',
 'ancient',
 'angel',
 'angeles',
 'animal',
 'anna',
 'answer',
 'apart',
 'apartment',
 'appear',
 'approach',
 'area',
 'arm',
 'army',
 'arrest',
 'arrival',
 'arrive',
 'art',
 'arthur',
 'artist',
 'ask',
 'aspire',
 'assassin',
 'assassination',
 'assign',
 'assignment',
 'assistant',
 'astronaut',
 'attack',
 'attempt'

In [82]:
# Vetorize Crew, and Actor
def clean_and_format(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase everything
    text = text.lower()
    
    # 2. Remove periods entirely (so "j.j. abrams" becomes "jj abrams")
    text = text.replace('.', '')
    text = text.replace('\'', '')
    text = text.replace("-", "")
    text = text.replace(",", "")

    
    # 3. Replace hyphens, slashes, or other punctuation with spaces
    # text = re.sub(r'[^a-z0-9_\s]', ' ', text)
    
    # 4. Clean up double spaces or double underscores
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'_{2,}', '_', text)

    # 5. Remove - in names (e.g. kim min-ji to kim minji)
    
    return text.strip()
counter_staff = CountVectorizer( ngram_range=(1,3), max_features=3000, token_pattern=r"\b\w+\b")
staff_soup = (movie_df['crew'].apply(lambda x: " ".join(x)) + movie_df['cast'].apply(lambda x: " ".join(x))).apply(clean_and_format)
display(staff_soup[0])
staff_vector = counter_staff.fit_transform(staff_soup)

# staff_vector.shape


'writer_alec_sokolow producer_edwin_catmull producer_ralph_guggenheim producer_steve_jobs writer_joel_cohen writer_joss_whedon director_john_lasseter writer_andrew_stanton producer_bonnie_arnoldactor_tom_hanks actor_tim_allen actor_don_rickles actor_jim_varney actor_wallace_shawn actor_john_ratzenberger actor_annie_potts actor_john_morris actor_erik_von_detten actor_laurie_metcalf actor_r_lee_ermey actor_sarah_freeman actor_penn_jillette actor_jack_angel actor_spencer_aste actor_greg_berg actor_lisa_bradley actor_kendall_cunningham actor_debi_derryberry actor_cody_dorkin actor_bill_farmer actor_craig_good actor_gregory_grudt actor_danielle_judovits actor_sam_lasseter actor_brittany_levenbrown actor_sherry_lynn actor_scott_mcafee actor_mickie_mcgowan actor_ryan_odonohue actor_jeff_pidgeon actor_patrick_pinney actor_phil_proctor actor_jan_rabson actor_joe_ranft actor_andrew_stanton actor_shane_sweet'

In [83]:
list(counter_staff.get_feature_names_out())

['actor_aaron_eckhart',
 'actor_aaron_lustig',
 'actor_aasif_mandvi',
 'actor_abigail_breslin',
 'actor_abraham_benrubi',
 'actor_ac_peterson',
 'actor_adam_brody',
 'actor_adam_driver',
 'actor_adam_goldberg',
 'actor_adam_lefevre',
 'actor_adam_scott',
 'actor_addison_richards',
 'actor_adewale_akinnuoyeagbaje',
 'actor_adolf_hitler',
 'actor_adrian_martinez',
 'actor_adrien_brody',
 'actor_afemo_omilami',
 'actor_agustín_almodóvar',
 'actor_aida_turturro',
 'actor_aidan_quinn',
 'actor_ajay_naidu',
 'actor_akio_otsuka',
 'actor_al_bain',
 'actor_al_bridge',
 'actor_al_cerullo',
 'actor_al_ferguson',
 'actor_al_leong',
 'actor_al_pacino',
 'actor_al_sapienza',
 'actor_alan_alda',
 'actor_alan_arkin',
 'actor_alan_blumenfeld',
 'actor_alan_cumming',
 'actor_alan_napier',
 'actor_alan_rickman',
 'actor_alan_ruck',
 'actor_alan_tudyk',
 'actor_alanna_ubach',
 'actor_albert_brooks',
 'actor_albert_finney',
 'actor_alberto_morin',
 'actor_alec_baldwin',
 'actor_alec_guinness',
 'actor_ale

In [84]:
# # Find rows containing 'actor_kim_ki' (searching both the concatenated staff_soup and the original cast list)
# mask1 = staff_soup.str.contains(r'\brobert_downey_jr\b', regex=True, na=False)
# mask2 = movie_df['cast'].apply(lambda x: 'robert_downey' in x if isinstance(x, (list, tuple)) else False)
# mask = mask1 | mask2

# result = movie_df[mask]
# display(result)
# print(f"Found {len(result)} row(s). Indexes: {list(result.index)}")

In [85]:
# result.iloc[0]['cast']

In [86]:
# Vectorize 
counter_keywords = CountVectorizer(ngram_range=(1,3), max_features=3000)
keywords_vec = counter_keywords.fit_transform(keywords_df['keywords']) 

keywords_vec.shape

(15711, 3000)

In [87]:
list(counter_keywords.get_feature_names_out())

['11',
 '1453',
 '16th',
 '16th century',
 '17th',
 '17th century',
 '18th',
 '18th century',
 '1900',
 '1910s',
 '1920s',
 '1930',
 '1940',
 '1950s',
 '1950s 1960',
 '1960',
 '1970s',
 '1980s',
 '1990s',
 '19th',
 '19th century',
 '1st',
 '1st century',
 '2000s',
 '3d',
 '3d animation',
 '476',
 '476 1453',
 'abandon',
 'abandonment',
 'abduction',
 'abortion',
 'abroad',
 'absurd',
 'absurd hilarious',
 'abuse',
 'abusive',
 'abusive father',
 'accident',
 'accidental',
 'accidental death',
 'accusation',
 'accuse',
 'act',
 'action',
 'action and',
 'action and animation',
 'action hero',
 'action remake',
 'activism',
 'activist',
 'actor',
 'actress',
 'adaptation',
 'addict',
 'addiction',
 'adjacent',
 'admire',
 'admire adore',
 'admire adore ambiguous',
 'adolescence',
 'adopt',
 'adoption',
 'adore',
 'adore ambiguous',
 'adore amuse',
 'adult',
 'adult animation',
 'adult novel',
 'adulterous',
 'adultery',
 'adultery infidelity',
 'adventure',
 'adventurer',
 'advertising',

In [97]:
keywords_vec_df = pd.DataFrame.sparse.from_spmatrix(
    keywords_vec,
    columns=counter_keywords.get_feature_names_out()
)

In [88]:
user_rating = pd.read_csv("../data/filtered_movie_ratings_merged.csv")

In [90]:
from sklearn.model_selection import train_test_split

# Split user_rating data into train and test sets
train_ratings, test_ratings = train_test_split(user_rating, test_size=0.2, random_state=42)

print(f"Training set size: {len(train_ratings)}")
print(f"Test set size: {len(test_ratings)}")

Training set size: 25189646
Test set size: 6297412


In [ ]:
# Define Algorithn
def get_recommendation(user_review, tfidf_matrix):
    # user_profile = Σ (rating_i) * item_vector_i / Σ |rating_i|
    profile = None
    for r in user_review:
        # incremental build of normalized user profile: profile <- Σ r_i * v_i / Σ |r_i|
        # support items as dicts {'tmdbId'/'movieId'/'id', 'rating'} or (id/index, rating)
        if profile is None:
            n_feats = tfidf_matrix.shape[1]
            profile = np.zeros(n_feats, dtype=float)
            weight_sum = 0.0
            id_to_idx = pd.Series(movie_df.index.values, index=movie_df['id']).to_dict()

        # extract key and rating from r
        if isinstance(r, dict):
            rating = r.get('rating') or r.get('score') or 0.0
            key = r.get('tmdbId') or r.get('movieId') or r.get('id')
        elif isinstance(r, (list, tuple)) and len(r) >= 2:
            key, rating = r[0], r[1]
        else:
            continue

        # resolve to a row index in movie_df / tfidf_matrix
        idx = None
        try:
            if key in id_to_idx:
                idx = id_to_idx[key]
            elif isinstance(key, (int, np.integer)) and 0 <= int(key) < len(movie_df):
                idx = int(key)
            else:
                idx = id_to_idx.get(int(key), None)
        except Exception:
            continue
        if idx is None:
            continue

        # get item vector and update running weighted average
        item_vec = tfidf_matrix[idx].toarray().ravel()
        prev_w = weight_sum
        weight_sum = prev_w + abs(rating)
        if weight_sum == 0:
            continue
        # incremental average: new_profile = (prev_profile*prev_w + rating*item_vec) / weight_sum
        profile = (profile * prev_w + (rating * item_vec)) / weight_sum
    # Calculate cosine similarity between user profile and all movies
    user_profile_vector = np.array(profile).reshape(1, -1)
    similarities = cosine_similarity(user_profile_vector, tfidf_matrix)[0]
    
    # Get top N recommended movie indices
    top_indices = np.argsort(similarities)[::-1][:10]
    
    return movie_df.iloc[top_indices], similarities

In [94]:
# Get train and test reviews for userId 8607
user_id = 8607

user_train_reviews = train_ratings[train_ratings['userId'] == user_id]
user_test_reviews = test_ratings[test_ratings['userId'] == user_id]

print(f"User {user_id} train reviews: {len(user_train_reviews)}")
print(f"User {user_id} test reviews: {len(user_test_reviews)}")

display(user_train_reviews)
display(user_test_reviews)

,userId,movieId,rating,timestamp,tmdbId
1326419,8607,1080,2.0,1080113010,583
30289956,193210,64839,4.5,1390397753,12163
5725100,36321,457,4.0,940638225,5503
16041750,102148,3683,5.0,1190928159,11368
2895303,18545,3993,3.0,1119705773,10876
...,...,...,...,...,...
21081788,133978,52287,3.5,1179664902,1267
26858567,171295,3686,4.0,1273311471,1551
23327850,148485,2324,4.5,1287926334,637
16094478,102447,112,3.0,1149087968,33542


In [101]:
rec, sim = get_recommendation(train_ratings, keywords_vec_df)
rec

,budget,genres,id,revenue,runtime,title,year,weighted_rating,description,keywords,crew,cast,productions
15710,0,"[Drama, Romance]",23550,500930,106,His Secret Life,2001,7.239475,when Antonia 's husband Massimo be kill in a c...,aid affectation male homosexuality family angr...,"[writer_gianni_romoli, writer_ferzan_özpetek, ...","[actor_margherita_buy, actor_stefano_accorsi, ...","[production_les_films_balenciaga, production_r..."
15709,8000000,"[Comedy, Thriller, Mystery]",1958,42400000,111,8 Women,2002,6.883085,eight woman gather to celebrate Christmas in a...,daughter upper class father murder musical fam...,"[writer_françois_ozon, director_françois_ozon,...","[actor_catherine_deneuve, actor_isabelle_huppe...","[production_fidélité_productions, production_f..."
15708,30000000,"[Thriller, Crime]",9039,13414416,106,Trapped,2002,6.195109,when their daughter be abduct by experienced k...,ransom kidnap hostage psychopath telephone maniac,"[director_luis_mandoki, writer_greg_iles, prod...","[actor_charlize_theron, actor_courtney_love, a...","[production_senator_film, production_propagand..."
15707,35000000,"[Action, Adventure, Drama, Romance, War]",9093,29882645,132,The Four Feathers,2002,6.635344,a young british officer resign his post when h...,army escape bravery loyalty british empire isl...,"[writer_michael_schiffer, writer_hossein_amini...","[actor_heath_ledger, actor_wes_bentley, actor_...","[production_paramount_pictures, production_mir..."
15706,10000000,"[Comedy, Drama]",9034,38068353,98,The Banger Sisters,2002,5.745638,"in the late ' 60 , the self - proclaim belle o...",friendship tattoo rock star groupie past prom ...,"[director_bob_dolman, writer_bob_dolman, produ...","[actor_goldie_hawn, actor_susan_sarandon, acto...","[production_fox_searchlight_pictures, producti..."
15705,70000000,"[Action, Adventure, Thriller, Science Fiction]",10550,19924033,91,Ballistic: Ecks vs. Sever,2002,4.596874,"Jeremiah Ecks , an FBI agent , realize that he...",sniper martial art fight loss of love one spy ...,"[director_wych_kaosayananda, producer_oliver_h...","[actor_antonio_banderas, actor_lucy_liu, actor...","[production_franchise_pictures, production_chr..."
15704,0,"[Drama, Thriller]",575,13782896,120,The Experiment,2001,7.390432,20 volunteer agree to take part in a seemingly...,prison journalist rape prisoner experiment bas...,"[producer_marc_conrad, director_oliver_hirschb...","[actor_moritz_bleibtreu, actor_christian_berke...","[production_fanes_film, production_typhoon_fil..."
15703,0,"[Comedy, Drama, Romance]",19460,0,123,Son of the Bride,2001,7.310929,"at age 42 , Rafael Belvedere be have a crisis ...","heart attack marriage buenos aires , argentina...","[director_juan_josé_campanella, writer_fernand...","[actor_ricardo_darín, actor_héctor_alterio, ac...","[production_tornasol_media, production_jempsa,..."
15702,0,"[Comedy, Horror, Mystery]",22244,0,100,Society,1989,6.676696,Bill Whitney be worried that he be different t...,high school orgy transformation secret society...,"[producer_keizo_kabata, producer_terry_ogisu, ...","[actor_billy_warlock, actor_connie_danese, act...","[production_society_productions, production_wi..."
15701,0,[Drama],29698,0,94,Ratcatcher,1999,7.005090,James Gillespie be 12 year old . the world he ...,"scotland glasgow , scotland 1970s neighbor can...","[director_lynne_ramsay, producer_gavin_emerson...","[actor_william_eadie, actor_tommy_flanagan, ac...","[production_pathé, production_bbc_film, produc..."


In [93]:
def get_recommendation_explanation():
    pass